# FX.fact_ohlc — Simple CRUD Test

Insert, read, update, and delete a row from the existing `FX.fact_ohlc` table.

In [7]:
from datetime import datetime, timezone
from decimal import Decimal

from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector

connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

Connected: mssql+pyodbc://@rv-database-1.ctym72ljvrjq.ap-southeast-1.rds.amazonaws.com:1433/IMDR?Trusted_Connection=yes&driver=SQL+Server


## 1. CREATE — Insert a row

In [14]:
insert_sql = text("""
    INSERT INTO FX.fact_ohlc
        (ts, symbol, series, tenor, deal_type, pair_used,
         open_px, high_px, low_px, close_px, mid_px,
         mid_mean_px, mid_median_px, bid, ask, n_ticks)
    OUTPUT INSERTED.id
    VALUES
        (:ts, :symbol, :series, :tenor, :deal_type, :pair_used,
         :open_px, :high_px, :low_px, :close_px, :mid_px,
         :mid_mean_px, :mid_median_px, :bid, :ask, :n_ticks)
""")

params = dict(
    ts=datetime.now(timezone.utc),
    symbol="EURUSD",
    series="spot",
    tenor="ON",
    deal_type="outright",
    pair_used="EURUSD",
    open_px=Decimal("1.08400000"),
    high_px=Decimal("1.08600000"),
    low_px=Decimal("1.08200000"),
    close_px=Decimal("1.08450000"),
    mid_px=Decimal("1.08425000"),
    mid_mean_px=Decimal("1.08410000"),
    mid_median_px=Decimal("1.08420000"),
    bid=Decimal("1.08400000"),
    ask=Decimal("1.08450000"),
    n_ticks=150,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

Inserted row with id = 3


## 2. READ — Fetch the row back

In [10]:
import pandas as pd

read_sql = text("SELECT * FROM FX.fact_ohlc")

with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection())

print(f"Total rows: {len(df)}")
df

Total rows: 34


,id,ts,symbol,series,tenor,deal_type,pair_used,open_px,high_px,low_px,close_px,mid_px,mid_mean_px,mid_median_px,bid,ask,n_ticks,created_at
0,106,2026-03-09 11:00:00 +00:00,USDPHP,NDF_1M,1M,NDF,USDPHP,59.594650,59.620000,59.564975,59.620000,59.620000,59.596895,59.593973,59.594400,59.645600,36000,2026-03-09 12:35:27.5492222
1,107,2026-03-09 11:00:00 +00:00,USDPHP,SPOT,SPOT,NDF,USDPHP,59.446750,59.462875,59.411800,59.459650,59.459650,59.442137,59.440620,59.435750,59.483550,36000,2026-03-09 12:35:27.5682226
2,108,2026-03-09 11:00:00 +00:00,USDINR,NDF_1M,1M,NDF,USDINR,92.771700,92.783250,92.717000,92.735000,92.735000,92.756679,92.761250,92.733250,92.736750,36000,2026-03-09 12:35:27.5892199
3,109,2026-03-09 11:00:00 +00:00,USDINR,SPOT,SPOT,NDF,USDINR,92.336000,92.345000,92.279200,92.299000,92.299000,92.318474,92.321850,92.296500,92.301500,36000,2026-03-09 12:35:27.6082263
4,110,2026-03-09 11:00:00 +00:00,USDKRW,NDF_1M,1M,NDF,USDKRW,1488.635000,1489.625000,1487.365800,1487.670500,1487.670500,1488.259515,1488.062750,1487.565500,1487.775500,36000,2026-03-09 12:35:27.6282257
5,111,2026-03-09 11:00:00 +00:00,USDKRW,SPOT,SPOT,NDF,USDKRW,1489.750000,1490.780750,1488.495000,1488.830000,1488.830000,1489.399607,1489.196500,1488.730000,1488.930000,36000,2026-03-09 12:35:27.6462184
6,112,2026-03-09 11:00:00 +00:00,USDIDR,NDF_1M,1M,NDF,USDIDR,16984.182500,16985.032500,16971.572500,16979.750000,16979.750000,16978.951175,16979.477500,16978.000000,16981.500000,36000,2026-03-09 12:35:27.6662199
7,113,2026-03-09 11:00:00 +00:00,USDIDR,SPOT,SPOT,NDF,USDIDR,16960.180545,16961.325000,16946.800000,16956.325000,16956.325000,16955.149314,16955.625000,16954.575000,16958.075000,36000,2026-03-09 12:35:27.6842175
8,114,2026-03-09 11:00:00 +00:00,USDTWD,NDF_1M,1M,NDF,USDTWD,31.932650,31.967000,31.925500,31.953000,31.953000,31.941913,31.942612,31.950650,31.955350,36000,2026-03-09 12:35:27.7022215
9,115,2026-03-09 11:00:00 +00:00,USDTWD,SPOT,SPOT,NDF,USDTWD,31.859250,31.892968,31.851050,31.877025,31.877025,31.868132,31.868000,31.876275,31.877775,36000,2026-03-09 12:35:27.7222170


## 3. UPDATE — Change close_px

In [16]:
update_sql = text("""
    UPDATE FX.fact_ohlc
    SET close_px = :close_px
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"close_px": Decimal("1.09000000"), "id": inserted_id})

# Verify the update
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated close_px = {df['close_px'].iloc[0]}")
df

Updated close_px = 1.09


,id,ts,symbol,series,tenor,deal_type,pair_used,open_px,high_px,low_px,close_px,mid_px,mid_mean_px,mid_median_px,bid,ask,n_ticks,created_at
0,3,2026-03-09 09:25:05 +00:00,EURUSD,spot,ON,outright,EURUSD,1.084,1.086,1.082,1.09,1.08425,1.0841,1.0842,1.084,1.0845,150,2026-03-09 09:25:05.3396233


## 4. DELETE — Remove the test row

In [11]:
delete_sql = text("DELETE FROM FX.fact_ohlc")

with connector.session() as session:
    result = session.execute(delete_sql)

print(f"Deleted {result.rowcount} row(s)")

# Verify table is empty
with connector.session() as session:
    df = pd.read_sql(text("SELECT COUNT(*) AS cnt FROM FX.fact_ohlc"), session.connection())

print(f"Rows remaining: {df['cnt'].iloc[0]}")

Deleted 34 row(s)
Rows remaining: 0


In [18]:
connector.dispose()
print("Done — connection pool closed.")

Done — connection pool closed.
